# LaBSE Script-Agnosticism Probing

This notebook runs the full probing suite from *Diagnosing and Mitigating Script-Based
Representational Divergence in Multilingual LLMs* on **LaBSE** (`sentence-transformers/LaBSE`)
only, matching the exact experimental protocol of the original paper so results slot directly
into Tables 2–7 and Figures 1–2.

**Why LaBSE is a critical addition:**
LaBSE was trained with a translation-ranking objective on 109 languages — qualitatively
different from the masked-LM (XLM-R), span-corruption (ByT5, mT5) objectives already studied.
It is the most widely deployed cross-lingual retrieval encoder and its omission is the
first thing retrieval-focused reviewers will flag.

**Experiments covered:**
1. Layer-wise cosine similarity (→ Table 2, Figure 1)
2. Script cluster analysis / silhouette (→ Table 3, Figure 2)
3. MLM script preference (→ Table 4; LaBSE has an MLM head)
4. Cross-script sentence retrieval MRR (→ Tables 5 & 6)
5. Causal decomposition — token overlap correlation (→ Table 5, Section 5.3)
6. Intervention study — all five interventions on LaBSE (→ new Table 8)

**Architecture notes:**
- 12 transformer layers (vs XLM-R Large: 24, ByT5/mT5 Large: 24)
- Hidden size 768 (vs 1024 for the Large models)
- SentencePiece tokenizer, ~501k-token vocabulary
- Has a masked-LM head → Experiment 3 applies directly
- Native CLS pooling via the sentence-transformers wrapper, but we use mean pooling
  throughout to match the original paper's embedding protocol exactly

## 1. Installation

In [ ]:
!pip install -q \
    transformers datasets \
    opencc-python-reimplemented \
    umap-learn scikit-learn \
    matplotlib seaborn pandas tqdm \
    sentencepiece protobuf \
    turkicnlp cyrtranslit pykakasi nltk aksharamukha

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
print('Installation complete.')

## 2. Imports and output directory

In [ ]:
import os, re, json, time, shutil, hashlib, platform, unicodedata, warnings
from datetime import datetime, timezone
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from datasets import load_dataset
from nltk.tokenize import sent_tokenize
from sklearn.metrics import silhouette_score
import umap

import opencc
import cyrtranslit
import pykakasi
from aksharamukha import transliterate as _aksharamukha
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForMaskedLM
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Configuration

In [ ]:
# LaBSE only — slot results into the same tables as the paper.
# Model string: sentence-transformers/LaBSE
# Architecture: BERT-base, 12 layers, hidden=768, ~471M params
# Tokenizer: SentencePiece, ~501k vocab (much larger than XLM-R's 250k)
# Training objective: translation ranking (bidirectional dual encoder)
# This is architecturally different from all three existing models.
LABSE_MODEL = 'sentence-transformers/LaBSE'
MODEL_NAME  = 'labse'

# Dataset sizes — identical to the original paper for direct comparability.
N_WORD_PAIRS = 500
N_SENT_PAIRS = 2000
N_EVAL_PAIRS = 500
N_TRAIN_PAIRS = N_SENT_PAIRS - N_EVAL_PAIRS   # 1500

# Layer fractions — same as paper. LaBSE has 12 layers so these resolve to
# layers [1, 4, 7, 10, 12] rather than XLM-R's [1, 7, 13, 19, 24].
# This is expected and handled automatically by layer_indices().
LAYER_FRACS = [0.0, 0.25, 0.5, 0.75, 1.0]

RETRIEVAL_RANDOM_BASELINE = 1.0 / N_SENT_PAIRS

RUN_TS  = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
RUN_DIR = Path('/kaggle/working') / f'labse_run_{RUN_TS}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

for sub in ['corpus', 'embeddings', 'exp1', 'exp2', 'exp3',
            'exp4', 'exp5', 'exp6', 'figures']:
    (RUN_DIR / sub).mkdir(exist_ok=True)

print(f'Output directory      : {RUN_DIR}')
print(f'Model                 : {LABSE_MODEL}')
print(f'Sentences per language: {N_SENT_PAIRS}')
print(f'  training split      : {N_TRAIN_PAIRS}')
print(f'  eval split          : {N_EVAL_PAIRS}')
print(f'  retrieval pool size : {N_SENT_PAIRS}')
print(f'  random baseline MRR : {RETRIEVAL_RANDOM_BASELINE:.6f}')

In [ ]:
# Reproducibility manifest
import transformers as _tf, sklearn as _sk

MANIFEST = {
    'run_id'  : RUN_TS,
    'started' : datetime.now(timezone.utc).isoformat(),
    'config'  : {
        'model'        : LABSE_MODEL,
        'N_WORD_PAIRS' : N_WORD_PAIRS,
        'N_SENT_PAIRS' : N_SENT_PAIRS,
        'LAYER_FRACS'  : LAYER_FRACS,
        'DEVICE'       : DEVICE,
    },
    'environment': {
        'python'      : platform.python_version(),
        'torch'       : torch.__version__,
        'transformers': _tf.__version__,
        'sklearn'     : _sk.__version__,
        'cuda'        : torch.version.cuda,
        'gpu'         : torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu',
    },
    'files': [],
}

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def save(rel_path, data, kind='json', description=''):
    dest = RUN_DIR / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    if kind == 'json':
        with open(dest, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    elif kind == 'csv':
        data.to_csv(dest, index=False)
    elif kind == 'npz':
        np.savez_compressed(dest, **data)
    elif kind == 'pdf':
        data.savefig(dest, dpi=150, bbox_inches='tight')
    elif kind == 'copy':
        shutil.copy2(data, dest)
    else:
        raise ValueError(f'Unknown kind: {kind}')
    MANIFEST['files'].append({
        'rel_path'   : rel_path,
        'kind'       : kind,
        'description': description,
        'sha256'     : _sha256(dest),
        'size_bytes' : dest.stat().st_size,
        'saved_at'   : datetime.now(timezone.utc).isoformat(),
    })
    _flush()
    return dest

def _flush():
    with open(RUN_DIR / 'manifest.json', 'w', encoding='utf-8') as f:
        json.dump(MANIFEST, f, ensure_ascii=False, indent=2, default=str)

def savefig(name, fig=None, description=''):
    if fig is None:
        fig = plt.gcf()
    return save(f'figures/{name}', fig, kind='pdf', description=description)

_flush()
print('Manifest initialised.')

## 4. Dataset construction

Identical corpus construction to the original paper — same sources, same sizes, same
conversion tools. If you already ran the original notebook and have the corpus JSON files,
you can skip this section and load them directly (see the cell at the end of this section).

| Language | Script A source | Conversion tool |
|---|---|---|
| Kazakh | kk Wikipedia | turkicnlp |
| Serbian | sr Wikipedia | cyrtranslit |
| Chinese | zh Wikipedia | OpenCC s2t |
| Hindi–Urdu | OPUS-100 en-hi | aksharamukha |
| Uyghur | OPUS-100 en-ug | turkicnlp |
| Japanese | ja Wikipedia | pykakasi |

In [ ]:
kaz_tr         = Transliterator('kaz', Script.CYRILLIC, Script.LATIN)
ug_tr          = Transliterator('uig', Script.PERSO_ARABIC, Script.LATIN)
kaz_tr_reverse = Transliterator('kaz', Script.LATIN, Script.CYRILLIC)
ug_tr_reverse  = Transliterator('uig', Script.LATIN, Script.PERSO_ARABIC)
s2t            = opencc.OpenCC('s2t')
t2s            = opencc.OpenCC('t2s')
kks            = pykakasi.kakasi()

def kanji_to_hira(text):
    return ''.join(item['hira'] for item in kks.convert(text))

def contains_kanji(text):
    return any(unicodedata.name(c, '').startswith('CJK') for c in text)

def hi_to_ur(text):
    return _aksharamukha.process('Devanagari', 'Urdu', text)

print('Transliterators ready.')

In [ ]:
def fetch_wiki(lang, n, min_w=5, max_w=25, max_articles=5000):
    ds = load_dataset('wikimedia/wikipedia', f'20231101.{lang}',
                      split='train', streaming=True)
    sents, k = [], 0
    for article in ds:
        k += 1
        text = article.get('text', '')
        raw  = re.split(r'[。！？\n،؟]+', text) if lang in ('zh', 'ja', 'ug') \
               else sent_tokenize(text)
        for s in raw:
            s = s.strip()
            if min_w < len(s.split()) < max_w and len(s) > 10:
                sents.append(s)
        if len(sents) >= n or k >= max_articles:
            break
    return sents[:n]

def fetch_opus(lang, pair, n):
    ds = load_dataset('Helsinki-NLP/opus-100', pair,
                      split='train', streaming=True)
    sents = []
    for item in ds:
        s = item['translation'].get(lang, '').strip()
        if s and 3 < len(s.split()) < 40:
            sents.append(s)
        if len(sents) >= n:
            break
    return sents

def words_from_pairs(pairs, n, min_len=2):
    def dedup(it):
        seen = set()
        return [x for x in it if x not in seen and not seen.add(x)]
    wa = dedup(w for s, _ in pairs for w in s.split() if len(w) >= min_len)
    wb = dedup(w for _, s in pairs for w in s.split() if len(w) >= min_len)
    return wa[:n], wb[:n]

def persist(lang, pairs, words, scripts, source, tool, extra=None):
    wa, wb = words
    meta = {
        'language': lang, 'script_a': scripts[0], 'script_b': scripts[1],
        'n_pairs': len(pairs), 'n_words_a': len(wa), 'n_words_b': len(wb),
        'source': source, 'conversion_tool': tool,
        'built_at': datetime.now(timezone.utc).isoformat(),
    }
    if extra:
        meta.update(extra)
    save(f'corpus/{lang}_pairs.json',   [[a, b] for a, b in pairs],
         description=f'{lang} sentence pairs')
    save(f'corpus/{lang}_words_a.json', wa,
         description=f'{lang} script-A word list')
    save(f'corpus/{lang}_words_b.json', wb,
         description=f'{lang} script-B word list')
    save(f'corpus/{lang}_meta.json',    meta,
         description=f'{lang} corpus metadata')

print('Corpus utilities ready.')

In [ ]:
# ── Option A: build corpus from scratch (same as original notebook) ──────────
# Set LOAD_FROM_DISK = False to run this block.
# ── Option B: load from a previous run ───────────────────────────────────────
# Set LOAD_FROM_DISK = True and point CORPUS_DIR at your existing corpus folder.

LOAD_FROM_DISK = False
CORPUS_DIR     = Path('/kaggle/working/YOUR_PREVIOUS_RUN/corpus')  # edit if needed

LANG_CONFIGS = [
    ('kazakh',     ('Cyrillic', 'Latin'),            'kk Wikipedia 20231101', 'turkicnlp'),
    ('serbian',    ('Cyrillic', 'Latin'),            'sr Wikipedia 20231101', 'cyrtranslit'),
    ('chinese',    ('Simplified', 'Traditional'),    'zh Wikipedia 20231101', 'opencc s2t'),
    ('hindi_urdu', ('Devanagari', 'Nastaliq'),       'OPUS-100 en-hi',        'aksharamukha'),
    ('uyghur',     ('Arabic-script', 'ULY Latin'),   'OPUS-100 en-ug',        'turkicnlp uig'),
    ('japanese',   ('Kanji/Mixed', 'Hiragana'),      'ja Wikipedia 20231101', 'pykakasi'),
]

if LOAD_FROM_DISK:
    print('Loading corpus from disk...')
    DATASETS = {}
    for lang, scripts, _, _ in LANG_CONFIGS:
        pairs   = [tuple(p) for p in json.load(open(CORPUS_DIR / f'{lang}_pairs.json'))]
        words_a = json.load(open(CORPUS_DIR / f'{lang}_words_a.json'))
        words_b = json.load(open(CORPUS_DIR / f'{lang}_words_b.json'))
        DATASETS[lang] = {'pairs': pairs, 'words': (words_a, words_b), 'scripts': scripts}
        print(f'  {lang:<12} {len(pairs)} pairs loaded')
else:
    print('[1/6] Kazakh  — Cyrillic ↔ Latin')
    kaz_cyr   = fetch_wiki('kk', N_SENT_PAIRS)
    KAZ_PAIRS = [(s, kaz_tr.transliterate(s)) for s in kaz_cyr]
    KAZ_WORDS = words_from_pairs(KAZ_PAIRS, N_WORD_PAIRS)
    persist('kazakh', KAZ_PAIRS, KAZ_WORDS, ('Cyrillic','Latin'),
            'kk Wikipedia 20231101', 'turkicnlp')
    print(f'  {len(KAZ_PAIRS)} pairs')

    print('[2/6] Serbian — Cyrillic ↔ Latin')
    srb_cyr   = fetch_wiki('sr', N_SENT_PAIRS)
    SRB_PAIRS = [(s, cyrtranslit.to_latin(s, 'sr')) for s in srb_cyr]
    SRB_WORDS = words_from_pairs(SRB_PAIRS, N_WORD_PAIRS)
    persist('serbian', SRB_PAIRS, SRB_WORDS, ('Cyrillic','Latin'),
            'sr Wikipedia 20231101', 'cyrtranslit')
    print(f'  {len(SRB_PAIRS)} pairs')

    print('[3/6] Chinese — Simplified ↔ Traditional')
    zh_simp   = fetch_wiki('zh', N_SENT_PAIRS)
    ZH_PAIRS  = [(s, s2t.convert(s)) for s in zh_simp]
    ZH_WORDS  = words_from_pairs(ZH_PAIRS, N_WORD_PAIRS)
    ZH_DIV    = [any(cs != t2s.convert(ct) for cs, ct in zip(a, b))
                 for a, b in ZH_PAIRS]
    persist('chinese', ZH_PAIRS, ZH_WORDS, ('Simplified','Traditional'),
            'zh Wikipedia 20231101', 'opencc s2t', extra={'n_divergent': sum(ZH_DIV)})
    print(f'  {len(ZH_PAIRS)} pairs, {sum(ZH_DIV)} lexically divergent')

    print('[4/6] Hindi–Urdu — Devanagari ↔ Nastaliq')
    hi_raw = fetch_opus('hi', 'en-hi', N_SENT_PAIRS * 3)
    HI_PAIRS, n_fail = [], 0
    for hi in hi_raw:
        try:
            ur = hi_to_ur(hi)
        except Exception:
            n_fail += 1; continue
        if ur and ur != hi:
            HI_PAIRS.append((hi, ur))
        if len(HI_PAIRS) >= N_SENT_PAIRS:
            break
    HI_WORDS = words_from_pairs(HI_PAIRS, N_WORD_PAIRS)
    persist('hindi_urdu', HI_PAIRS, HI_WORDS, ('Devanagari','Nastaliq'),
            'OPUS-100 en-hi', 'aksharamukha', extra={'n_conversion_errors': n_fail})
    print(f'  {len(HI_PAIRS)} pairs')

    print('[5/6] Uyghur  — Arabic ↔ ULY Latin')
    ug_raw    = fetch_opus('ug', 'en-ug', N_SENT_PAIRS * 3)
    ug_source = 'OPUS-100 en-ug'
    if len(ug_raw) < 10:
        ug_raw    = fetch_wiki('ug', N_SENT_PAIRS * 3)
        ug_source = 'ug Wikipedia 20231101'
    UG_PAIRS = []
    for arab in ug_raw:
        try:
            lat = ug_tr.transliterate(arab)
        except Exception:
            continue
        if sum(1 for c in lat if ord(c) < 256) / max(len(lat), 1) > 0.6:
            UG_PAIRS.append((arab, lat))
        if len(UG_PAIRS) >= N_SENT_PAIRS:
            break
    UG_WORDS = words_from_pairs(UG_PAIRS, N_WORD_PAIRS)
    persist('uyghur', UG_PAIRS, UG_WORDS, ('Arabic-script','ULY Latin'),
            ug_source, 'turkicnlp uig')
    print(f'  {len(UG_PAIRS)} pairs')

    print('[6/6] Japanese — Kanji/Mixed ↔ Hiragana')
    ja_raw   = fetch_wiki('ja', N_SENT_PAIRS * 4)
    JA_PAIRS = []
    for s in ja_raw:
        if not contains_kanji(s): continue
        try:
            h = kanji_to_hira(s)
            if len(h) > 5:
                JA_PAIRS.append((s, h))
        except Exception:
            pass
        if len(JA_PAIRS) >= N_SENT_PAIRS:
            break
    JA_WORDS = words_from_pairs(JA_PAIRS, N_WORD_PAIRS)
    persist('japanese', JA_PAIRS, JA_WORDS, ('Kanji/Mixed','Hiragana'),
            'ja Wikipedia 20231101', 'pykakasi')
    print(f'  {len(JA_PAIRS)} pairs')

    DATASETS = {
        'kazakh'    : {'pairs': KAZ_PAIRS, 'words': KAZ_WORDS, 'scripts': ('Cyrillic','Latin')},
        'serbian'   : {'pairs': SRB_PAIRS, 'words': SRB_WORDS, 'scripts': ('Cyrillic','Latin')},
        'chinese'   : {'pairs': ZH_PAIRS,  'words': ZH_WORDS,  'scripts': ('Simplified','Traditional')},
        'uyghur'    : {'pairs': UG_PAIRS,  'words': UG_WORDS,  'scripts': ('Arabic-script','ULY Latin')},
        'japanese'  : {'pairs': JA_PAIRS,  'words': JA_WORDS,  'scripts': ('Kanji/Mixed','Hiragana')},
        'hindi_urdu': {'pairs': HI_PAIRS,  'words': HI_WORDS,  'scripts': ('Devanagari','Nastaliq')},
    }

print()
print(f'{"Language":<12}  {"Pairs":>6}  {"Words":>6}')
print('-' * 28)
for lang, d in DATASETS.items():
    print(f'{lang:<12}  {len(d["pairs"]):>6}  {len(d["words"][0]):>6}')

## 5. Load LaBSE

We load LaBSE once and keep it in memory for all experiments. Unlike the original
notebook which loads/unloads three models sequentially, LaBSE at 471M parameters
fits comfortably alongside the embedding cache on a 16GB GPU.

**Pooling note:** The sentence-transformers LaBSE checkpoint includes a mean-pooling
layer and a normalisation layer on top of BERT. We bypass those and use our own
mean pooling on the raw hidden states, exactly matching the protocol applied to
XLM-R, ByT5, and mT5 in the original notebook.

In [ ]:
print(f'Loading {LABSE_MODEL}...')
tok  = AutoTokenizer.from_pretrained(LABSE_MODEL)
mod  = AutoModel.from_pretrained(LABSE_MODEL).to(DEVICE)
mod.eval()

N_LAYERS    = mod.config.num_hidden_layers   # 12 for LaBSE
HIDDEN_SIZE = mod.config.hidden_size         # 768 for LaBSE
FINAL_LAYER = N_LAYERS

print(f'Loaded successfully.')
print(f'  num_hidden_layers : {N_LAYERS}')
print(f'  hidden_size       : {HIDDEN_SIZE}')
print(f'  vocab_size        : {mod.config.vocab_size}')
print(f'  Parameters        : {sum(p.numel() for p in mod.parameters()) / 1e6:.0f}M')

# Resolve LAYER_FRACS to concrete layer indices for LaBSE's 12-layer architecture.
# Index 0 = embedding layer; 1..12 = transformer block outputs.
# Fracs [0.0, 0.25, 0.5, 0.75, 1.0] → layers [1, 4, 7, 10, 12]
LAYER_IDXS = [max(1, min(N_LAYERS, 1 + int(round(f * (N_LAYERS - 1)))))
              for f in LAYER_FRACS]
print(f'  Layer indices probed : {LAYER_IDXS}  (fracs={LAYER_FRACS})')

## 6. Embedding utilities

In [ ]:
def embed(texts, idxs=None, batch_size=32):
    """
    Mean-pool hidden states at each requested layer index.
    Returns dict {layer_idx -> np.ndarray of shape (N, H)}.

    LaBSE is BERT-base so we can afford larger batches than
    the Large models in the original notebook.
    """
    if idxs is None:
        idxs = LAYER_IDXS
    buckets = defaultdict(list)
    for i in range(0, len(texts), batch_size):
        batch = texts[i: i + batch_size]
        try:
            enc = tok(batch, return_tensors='pt', padding=True,
                      truncation=True, max_length=128).to(DEVICE)
            with torch.no_grad():
                out = mod(**enc, output_hidden_states=True)
            hs   = out.hidden_states          # tuple of (B, T, H), len = N_LAYERS+1
            mask = enc['attention_mask'].unsqueeze(-1).float()
            for li in idxs:
                h      = hs[li]
                pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
                pooled = F.normalize(pooled, dim=-1)
                buckets[li].append(pooled.cpu().float().numpy())
        except Exception as e:
            if not hasattr(embed, '_warned'):
                print(f'  batch {i//batch_size} failed: {e}')
                embed._warned = True
    if not buckets:
        return {li: np.zeros((len(texts), HIDDEN_SIZE)) for li in idxs}
    return {li: np.concatenate(arrs) for li, arrs in buckets.items()}


def embed_all_layers(texts, batch_size=32):
    """Single forward pass returning all N_LAYERS+1 hidden states (for layer search)."""
    all_idxs = list(range(N_LAYERS + 1))
    return embed(texts, idxs=all_idxs, batch_size=batch_size)


def paired_cos(a, b):
    """Row-wise cosine similarity, returns (N,) array."""
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-9)
    b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-9)
    return (a * b).sum(axis=1)


def compute_mrr(q, k, correct_idx):
    """Mean Reciprocal Rank. q (N,D), k (M,D). Returns (mrr, per_query_list)."""
    qn = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-9)
    kn = k / (np.linalg.norm(k, axis=1, keepdims=True) + 1e-9)
    S  = qn @ kn.T
    rr, per_query = 0.0, []
    for i, ci in enumerate(correct_idx):
        rank = int(np.where(np.argsort(-S[i]) == ci)[0][0]) + 1
        rr  += 1.0 / rank
        per_query.append({'query_idx': i, 'rank': rank, 'rr': 1.0 / rank})
    return rr / len(correct_idx), per_query


def save_emb(emb_dict, lang, tag):
    arrays = {f'layer_{li}': arr for li, arr in emb_dict.items()}
    save(f'embeddings/{lang}_{tag}.npz', arrays, kind='npz',
         description=f'LaBSE embeddings — {lang} ({tag})')


print('Embedding utilities ready.')

## 7. Experiment 1 — Layer-wise cosine similarity

**Paper target:** Table 2 (add LaBSE column) and Figure 1 (add LaBSE panel).

LaBSE has 12 layers vs 24 for the Large models, so the absolute layer indices
differ but the fractional positions are comparable. The key question is whether
LaBSE's translation-ranking fine-tuning produces flatter or steeper similarity
trajectories than XLM-R's masked-LM objective.

In [ ]:
print('Experiment 1: layer-wise cosine similarity\n')
layerwise = {}

for lang, d in DATASETS.items():
    wa, wb = list(d['words'][0]), list(d['words'][1])
    n      = min(len(wa), len(wb))
    if n == 0:
        continue
    wa, wb = wa[:n], wb[:n]

    ea = embed(wa)
    eb = embed(wb)
    save_emb(ea, lang, 'words_a')
    save_emb(eb, lang, 'words_b')

    res = {}
    for li, frac in zip(LAYER_IDXS, LAYER_FRACS):
        sims    = paired_cos(ea[li], eb[li])
        res[frac] = {'mean': float(np.nanmean(sims)),
                     'std' : float(np.nanstd(sims)),
                     'n'   : n}
    layerwise[lang] = res

    row = '  '.join(f'L{int(f*100):3d}%={v["mean"]:.3f}' for f, v in res.items())
    print(f'  {lang:<12} {row}')

save('exp1/labse_layerwise.json', layerwise,
     description='Exp 1 layer-wise cosine similarity — LaBSE')
print('\nExperiment 1 complete.')

In [ ]:
# Figure 1 panel for LaBSE (add alongside XLM-R / ByT5 / mT5 panels in the paper)
lang_colors = {
    'kazakh':'#1f77b4', 'serbian':'#ff7f0e', 'chinese':'#2ca02c',
    'uyghur':'#9467bd', 'japanese':'#8c564b', 'hindi_urdu':'#d62728',
}
lang_labels = {
    'kazakh':'Kazakh', 'serbian':'Serbian', 'chinese':'Chinese',
    'uyghur':'Uyghur', 'japanese':'Japanese', 'hindi_urdu':'Hindi–Urdu',
}

fig, ax = plt.subplots(figsize=(6, 4.5))
for lang, res in layerwise.items():
    xs = list(res.keys())
    ys = [v['mean'] for v in res.values()]
    es = [v['std']  for v in res.values()]
    ax.plot(xs, ys, marker='o', label=lang_labels[lang],
            color=lang_colors[lang], linewidth=1.8)
    ax.fill_between(xs, [y-e for y,e in zip(ys,es)],
                        [y+e for y,e in zip(ys,es)],
                    alpha=0.12, color=lang_colors[lang])
ax.set_title('LaBSE', fontweight='bold')
ax.set_xlabel('Layer depth')
ax.set_xticks(LAYER_FRACS)
ax.set_xticklabels(['0%','25%','50%','75%','100%'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Mean paired cosine similarity (±1 SD)')
ax.legend(fontsize=8)
fig.suptitle('Layer-wise script similarity — LaBSE', fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('exp1_layerwise_labse.pdf', fig, description='Exp 1 figure — LaBSE')
plt.show()

# Print Table 2 rows for copy-paste into the paper
print('\n=== Table 2 rows (LaBSE) ===')
print(f'{"Language":<12}  ' + '  '.join(f'{int(f*100):3d}%' for f in LAYER_FRACS))
for lang, res in layerwise.items():
    vals = '  '.join(f'{v["mean"]:.3f}' for v in res.values())
    print(f'{lang:<12}  {vals}')

## 8. Experiment 2 — Script cluster analysis

**Paper target:** Table 3 (add LaBSE column) and Figure 2 (add LaBSE row of UMAP plots).

Silhouette scores near 0 indicate script-agnostic representations — the desired
property. LaBSE's translation objective should push it toward lower silhouette
scores than XLM-R for well-resourced languages.

In [ ]:
print('Experiment 2: script cluster analysis\n')
cluster_res = {}

for lang, d in DATASETS.items():
    sa     = [p[0] for p in d['pairs'][:N_SENT_PAIRS]]
    sb     = [p[1] for p in d['pairs'][:N_SENT_PAIRS]]
    labels = [0]*len(sa) + [1]*len(sb)

    E_dict = embed(sa + sb, idxs=[FINAL_LAYER])
    E      = E_dict[FINAL_LAYER]
    sil    = silhouette_score(E, labels)

    save_emb({FINAL_LAYER: E}, lang, 'sents')
    cluster_res[lang] = {'embeddings': E, 'labels': labels, 'silhouette': sil}
    print(f'  {lang:<12} silhouette = {sil:.3f}')

save('exp2/labse_silhouette.json',
     {l: r['silhouette'] for l, r in cluster_res.items()},
     description='Exp 2 silhouette scores — LaBSE')
print('\nExperiment 2 complete.')

In [ ]:
# UMAP visualisation — one row of panels for Figure 2
reducer    = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
script_pal = ['#4292c6','#ef6548']
n_l        = len(DATASETS)
fig, axes  = plt.subplots(1, n_l, figsize=(4.5*n_l, 3.5))
umap_coords = {}

for li, (lang, res) in enumerate(cluster_res.items()):
    ax  = axes[li]
    E2  = reducer.fit_transform(res['embeddings'])
    sl  = np.array(res['labels'])
    umap_coords[lang] = E2.tolist()
    for sc, sc_name in enumerate(DATASETS[lang]['scripts']):
        m = sl == sc
        ax.scatter(E2[m,0], E2[m,1], c=script_pal[sc],
                   alpha=0.45, s=8, label=sc_name)
    ax.set_title(f'LaBSE | {lang}\nsil={res["silhouette"]:.3f}', fontsize=8)
    ax.legend(fontsize=6, markerscale=2)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('UMAP of LaBSE sentence embeddings (coloured by script)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('exp2_umap_labse.pdf', fig, description='Exp 2 UMAP — LaBSE')
plt.show()

save('exp2/labse_umap_coordinates.json', umap_coords,
     description='Exp 2 UMAP 2D coordinates — LaBSE')

# Print Table 3 column for copy-paste
print('\n=== Table 3 column (LaBSE) ===')
for lang, res in cluster_res.items():
    print(f'  {lang:<12} {res["silhouette"]:.3f}')

## 9. Experiment 3 — MLM script preference

**Paper target:** Table 4 (add LaBSE column).

LaBSE is built on mBERT with a masked language modeling head, so this experiment
applies directly. LaBSE's large vocabulary (501k tokens) means fewer UNK skips
than XLM-R (250k), so we expect higher coverage, especially for low-resource scripts.
The comparison between XLM-R and LaBSE script preference scores tests whether the
translation-ranking fine-tuning shifts the model's implicit script bias.

Japanese and Chinese are excluded for the same reasons as in the original paper
(no whitespace word boundaries; many shared characters).

In [ ]:
print('Loading LaBSE MLM head for Experiment 3...')
# AutoModelForMaskedLM loads the same weights as AutoModel but with the
# language model head attached. We reload rather than reuse mod because
# AutoModel does not expose the LM head logits.
mod_mlm = AutoModelForMaskedLM.from_pretrained(LABSE_MODEL).to(DEVICE)
mod_mlm.eval()
print('MLM model loaded.')

MLM_LANGS = ['kazakh', 'serbian', 'uyghur', 'hindi_urdu']  # same exclusions as paper
mlm_results = {}

for lang in MLM_LANGS:
    d            = DATASETS[lang]
    wa, wb       = list(d['words'][0])[:500], list(d['words'][1])[:500]
    per_pair_log = []
    prefer_a, total = 0, 0

    for word_a, word_b in zip(wa, wb):
        ta = tok.tokenize(word_a)
        tb = tok.tokenize(word_b)
        if not ta or not tb:
            continue

        text = tok.convert_tokens_to_string([tok.mask_token] + ta[1:])
        enc  = tok(text, return_tensors='pt',
                   truncation=True, max_length=64).to(DEVICE)
        mpos = (enc['input_ids'] == tok.mask_token_id).nonzero(as_tuple=True)
        if len(mpos[0]) == 0:
            continue

        id_a = tok.convert_tokens_to_ids(ta[0])
        id_b = tok.convert_tokens_to_ids(tb[0])
        if id_a == tok.unk_token_id or id_b == tok.unk_token_id:
            continue

        with torch.no_grad():
            logits = mod_mlm(**enc).logits[0, mpos[1][0].item()]

        la, lb    = float(logits[id_a].cpu()), float(logits[id_b].cpu())
        pa        = la > lb
        prefer_a += int(pa)
        total    += 1
        per_pair_log.append({
            'word_a': word_a, 'word_b': word_b,
            'token_a': ta[0], 'token_b': tb[0],
            'logit_a': la,    'logit_b': lb,
            'prefers_a': pa,
        })

    pref = prefer_a / total if total > 0 else float('nan')
    mlm_results[lang] = {'prefer_a': pref, 'n': total, 'n_unk_skipped': len(wa) - total}
    save(f'exp3/labse_{lang}_per_pair.json', per_pair_log,
         description=f'Exp 3 MLM per-pair logits — LaBSE {lang}')
    print(f'  {lang:<12} prefers {d["scripts"][0]} = {pref:.1%}  '
          f'(n={total}, skipped_unk={len(wa)-total})')

save('exp3/labse_aggregate.json', mlm_results,
     description='Exp 3 MLM aggregate script preference — LaBSE')

# Free MLM model; we don't need it again
del mod_mlm; torch.cuda.empty_cache()

print('\n=== Table 4 column (LaBSE) ===')
for lang, res in mlm_results.items():
    print(f'  {lang:<12} Script A pref = {res["prefer_a"]:.1%}  (n={res["n"]})')
print('\nExperiment 3 complete.')

## 10. Experiment 4 — Cross-script sentence retrieval MRR

**Paper target:** Tables 5 & 6 (add LaBSE row/column), Section 5.3 (recompute
token-overlap correlation including LaBSE).

This is the most important experiment for the paper's reviewer concerns.
LaBSE's translation-ranking objective directly optimises for cross-lingual
sentence similarity, so it should substantially outperform XLM-R on retrieval
for well-resourced pairs. The question is whether it also recovers performance
for low-resource digraphic pairs like Kazakh, Uyghur, and Hindi–Urdu.

In [ ]:
print('Experiment 4: cross-script sentence retrieval MRR\n')
retrieval_results = {}
token_overlaps    = {}

for lang, d in DATASETS.items():
    eval_pairs  = d['pairs'][-N_EVAL_PAIRS:]
    sents_a     = [p[0] for p in eval_pairs]
    all_b_sents = [p[1] for p in d['pairs']]
    correct_idx = list(range(N_SENT_PAIRS - N_EVAL_PAIRS, N_SENT_PAIRS))

    ea = embed(sents_a,      idxs=[FINAL_LAYER])[FINAL_LAYER]
    ek = embed(all_b_sents,  idxs=[FINAL_LAYER])[FINAL_LAYER]
    save_emb({FINAL_LAYER: ea}, lang, 'retrieval_q')
    save_emb({FINAL_LAYER: ek}, lang, 'retrieval_k')

    mrr, pq = compute_mrr(ea, ek, correct_idx)
    retrieval_results[lang] = mrr
    save(f'exp4/labse_{lang}_per_query.json', pq,
         description=f'Exp 4 per-query ranks — LaBSE {lang}')

    # Token overlap — Jaccard similarity between LaBSE tokenisations of both scripts
    # LaBSE has ~501k vocab so overlaps may differ from XLM-R's 250k-vocab overlaps.
    overlaps = []
    for wa, wb in zip(list(d['words'][0])[:100], list(d['words'][1])[:100]):
        ta = set(tok.tokenize(wa))
        tb = set(tok.tokenize(wb))
        if ta | tb:
            overlaps.append(len(ta & tb) / len(ta | tb))
    tok_ov = float(np.mean(overlaps)) if overlaps else float('nan')
    token_overlaps[lang] = tok_ov

    print(f'  {lang:<12} MRR = {mrr:.4f}  tok_overlap = {tok_ov:.3f}')

save('exp4/labse_mrr.json', retrieval_results,
     description='Exp 4 MRR — LaBSE')
save('exp4/labse_token_overlaps.json', token_overlaps,
     description='Exp 4 token overlaps (LaBSE tokenizer) — all languages')
print('\nExperiment 4 complete.')

In [ ]:
# Compute token-overlap → MRR correlation (Section 5.3)
# The paper reports r=0.728–0.732 for the three original models.
# If LaBSE's correlation is similar, it supports the model-agnostic diagnostic claim.

from scipy.stats import pearsonr

ov_vals  = [token_overlaps[l]       for l in DATASETS]
mrr_vals = [retrieval_results[l]    for l in DATASETS]
fl_sims  = [layerwise[l][max(layerwise[l].keys())]['mean'] for l in DATASETS]

r_mrr, p_mrr = pearsonr(ov_vals, mrr_vals)
r_sim, p_sim = pearsonr(ov_vals, fl_sims)

print('=== Section 5.3: Token overlap correlations (LaBSE) ===')
print(f'  r(overlap, MRR)      = {r_mrr:.3f}  (p={p_mrr:.3f})')
print(f'  r(overlap, FL cosim) = {r_sim:.3f}  (p={p_sim:.3f})')
print(f'  (Paper reports r=0.728–0.732 for MRR, r=0.919–0.981 for cosim)')

save('exp4/labse_overlap_correlations.json',
     {'r_overlap_mrr': r_mrr, 'p_overlap_mrr': p_mrr,
      'r_overlap_sim': r_sim, 'p_overlap_sim': p_sim},
     description='Exp 4 token-overlap correlations — LaBSE')

# Print Tables 5 & 6 rows for copy-paste
print('\n=== Table 5 row (LaBSE) ===')
print(f'{"Language":<12}  {"FL cosim":>9}  {"MRR":>6}  {"Tok. overlap":>12}')
for lang in DATASETS:
    fl = layerwise[lang][max(layerwise[lang].keys())]['mean']
    print(f'{lang:<12}  {fl:>9.3f}  {retrieval_results[lang]:>6.3f}  {token_overlaps[lang]:>12.3f}')

print('\n=== Table 6 row (LaBSE) ===')
for lang, mrr in retrieval_results.items():
    print(f'  {lang:<12} {mrr:.3f}')
print(f'  {"Mean":<12} {np.mean(list(retrieval_results.values())):.3f}')

## 11. Experiment 5 — Causal decomposition

**Paper target:** Table 5 extended, Section 5.1 and 5.2 discussion.

LaBSE adds a fourth data point to the architecture comparison. Unlike XLM-R
(masked LM) and ByT5/mT5 (span corruption), LaBSE uses a translation-ranking
objective — making it a qualitatively distinct point in the objective × tokenization
space. The Kazakh–Serbian gap on LaBSE isolates whether the data-volume effect
observed in the original paper persists under a retrieval-oriented objective.

In [ ]:
# The causal decomposition compares Kazakh and Serbian, which share a near-bijective
# Cyrillic–Latin mapping but differ substantially in pretraining data volume.
# A large silhouette/MRR gap between them on LaBSE replicates the data-volume finding.

kaz_sil = cluster_res['kazakh']['silhouette']
srb_sil = cluster_res['serbian']['silhouette']
kaz_mrr = retrieval_results['kazakh']
srb_mrr = retrieval_results['serbian']
kaz_fl  = layerwise['kazakh'][max(layerwise['kazakh'].keys())]['mean']
srb_fl  = layerwise['serbian'][max(layerwise['serbian'].keys())]['mean']

data_vol_sil = kaz_sil - srb_sil   # positive = more script-separated for Kazakh
data_vol_mrr = srb_mrr - kaz_mrr   # positive = better retrieval for Serbian

print('=== Section 5.2: Data volume effect (LaBSE) ===')
print(f'  Kazakh  silhouette = {kaz_sil:.3f},  MRR = {kaz_mrr:.3f},  FL cosim = {kaz_fl:.3f}')
print(f'  Serbian silhouette = {srb_sil:.3f},  MRR = {srb_mrr:.3f},  FL cosim = {srb_fl:.3f}')
print(f'  Gap (Kazakh−Serbian) silhouette : {data_vol_sil:+.3f}')
print(f'  Gap (Serbian−Kazakh) MRR        : {data_vol_mrr:+.3f}')
print(f'  (Paper reports XLM-R gaps: silhouette=0.230, MRR=0.657)')

# Full decomposition table
decomp_rows = []
for lang in DATASETS:
    fl  = layerwise[lang][max(layerwise[lang].keys())]['mean']
    decomp_rows.append({
        'Language'    : lang,
        'FL cosim'    : round(fl, 3),
        'MRR'         : round(retrieval_results[lang], 3),
        'Silhouette'  : round(cluster_res[lang]['silhouette'], 3),
        'Tok. overlap': round(token_overlaps[lang], 3),
    })
decomp_df = pd.DataFrame(decomp_rows)
print('\n=== LaBSE full comparison table ===')
print(decomp_df.to_string(index=False))

save('exp5/labse_decomposition.csv', decomp_df, kind='csv',
     description='Exp 5 decomposition table — LaBSE')
save('exp5/labse_decomposition.json',
     {'table': decomp_rows,
      'data_volume_sil_gap': data_vol_sil,
      'data_volume_mrr_gap': data_vol_mrr},
     description='Exp 5 causal decomposition — LaBSE')
print('\nExperiment 5 complete.')

## 12. Experiment 6 — Intervention study

**Paper target:** New Table 8 (LaBSE intervention results).

The paper's intervention study (Table 7) covers XLM-R only. Running the same
five interventions on LaBSE tests whether they generalise to a model that already
has a stronger retrieval baseline. Key questions:

- Does whitening still help if LaBSE's representations are less anisotropic than XLM-R?
- Does normalisation provide the same near-perfect gains for Kazakh and Uyghur?
- Does contrastive alignment degrade Chinese (which is near ceiling)?

Interventions applied (identical protocol to the paper):
1. Script normalisation
2. Whitening (fit on 1,500 training pairs)
3. CSLS retrieval
4. Layer search
5. Contrastive alignment (NT-Xent, 1,500 pairs, 10 epochs)

In [ ]:
# Normalizers — identical to the original notebook
NORMALIZERS = {
    'kazakh'    : kaz_tr_reverse.transliterate,
    'serbian'   : lambda s: cyrtranslit.to_cyrillic(s, 'sr'),
    'chinese'   : t2s.convert,
    'uyghur'    : ug_tr_reverse.transliterate,
    'japanese'  : kanji_to_hira,
    'hindi_urdu': lambda s: _aksharamukha.process('Urdu', 'Devanagari', s),
}

def whiten_fit(X_train):
    mu  = X_train.mean(axis=0, keepdims=True)
    Xc  = X_train - mu
    U, S, _ = np.linalg.svd(np.cov(Xc, rowvar=False))
    W   = U @ np.diag(1.0 / np.sqrt(S + 1e-5))
    return mu, W

def whiten_apply(X, mu, W):
    return (X - mu) @ W

def csls_sim(x, y, k=10):
    sim  = x @ y.T
    r_x  = np.mean(np.sort(sim, axis=1)[:, -k:], axis=1)
    r_y  = np.mean(np.sort(sim, axis=0)[-k:, :], axis=0)
    return 2 * sim - r_x[:, None] - r_y[None, :]

def mrr_csls(q, k, correct_idx):
    qn, kn = (v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9) for v in (q, k))
    S  = csls_sim(qn, kn)
    rr = [1.0 / (np.where(np.argsort(-S[i]) == ci)[0][0] + 1)
          for i, ci in enumerate(correct_idx)]
    return float(np.mean(rr))

class PairDataset(Dataset):
    def __init__(self, a, b):
        self.a = torch.tensor(a, dtype=torch.float32)
        self.b = torch.tensor(b, dtype=torch.float32)
    def __len__(self):        return len(self.a)
    def __getitem__(self, i): return self.a[i], self.b[i]

class ProjectionHead(nn.Module):
    # Input dim is 768 for LaBSE (vs 1024 for XLM-R Large)
    def __init__(self, d_in=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 512), nn.ReLU(), nn.Linear(512, 256)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

def nt_xent(z1, z2, tau=0.05):
    logits = (z1 @ z2.T) / tau
    labels = torch.arange(len(z1), device=z1.device)
    return (F.cross_entropy(logits, labels) +
            F.cross_entropy(logits.T, labels)) / 2

print('Intervention utilities ready.')

In [ ]:
print('Experiment 6: intervention study on LaBSE\n')
int_results = {}

for lang, d in DATASETS.items():
    print(f'\n{"="*60}\n{lang.upper()}\n{"="*60}')

    train_pairs = d['pairs'][:N_TRAIN_PAIRS]
    eval_pairs  = d['pairs'][-N_EVAL_PAIRS:]
    sents_a     = [p[0] for p in eval_pairs]
    sents_b     = [p[1] for p in eval_pairs]
    all_b_sents = [p[1] for p in d['pairs']]
    correct_idx = list(range(N_SENT_PAIRS - N_EVAL_PAIRS, N_SENT_PAIRS))

    # ── Baseline ────────────────────────────────────────────────────────────
    print('Baseline...')
    ea = embed(sents_a,      idxs=[FINAL_LAYER])[FINAL_LAYER]
    eb = embed(all_b_sents,  idxs=[FINAL_LAYER])[FINAL_LAYER]
    mrr_base, pq_base = compute_mrr(ea, eb, correct_idx)
    print(f'  Baseline MRR: {mrr_base:.3f}')

    # ── Script normalisation ─────────────────────────────────────────────────
    print('Normalisation...')
    try:
        norm       = NORMALIZERS[lang]
        key_pool_n = [norm(s) for s in all_b_sents]
        eb_n       = embed(key_pool_n, idxs=[FINAL_LAYER])[FINAL_LAYER]
        mrr_norm, _ = compute_mrr(ea, eb_n, correct_idx)
    except Exception as e:
        print(f'  normalisation failed: {e}')
        mrr_norm = float('nan')
    print(f'  Normalised MRR: {mrr_norm:.3f}')

    # ── Whitening ────────────────────────────────────────────────────────────
    print('Whitening...')
    train_sa = [p[0] for p in train_pairs]
    train_sb = [p[1] for p in train_pairs]
    train_ea = embed(train_sa, idxs=[FINAL_LAYER])[FINAL_LAYER]
    train_eb = embed(train_sb, idxs=[FINAL_LAYER])[FINAL_LAYER]
    mu_w, W_w = whiten_fit(np.concatenate([train_ea, train_eb]))
    mrr_white, _ = compute_mrr(whiten_apply(ea, mu_w, W_w),
                               whiten_apply(eb, mu_w, W_w), correct_idx)
    print(f'  Whitened MRR: {mrr_white:.3f}')

    # ── CSLS ─────────────────────────────────────────────────────────────────
    print('CSLS...')
    mrr_csls_val = mrr_csls(ea, eb, correct_idx)
    print(f'  CSLS MRR: {mrr_csls_val:.3f}')

    # ── Layer search ─────────────────────────────────────────────────────────
    # Single forward pass over all 12+1 layers — faster than the Large models
    # in the original notebook since LaBSE has half the depth.
    print('Layer search...')
    q_by_layer = embed_all_layers(sents_a)
    k_by_layer = embed_all_layers(all_b_sents)
    layer_mrrs = []
    for li in range(N_LAYERS + 1):
        m, _ = compute_mrr(q_by_layer[li], k_by_layer[li], correct_idx)
        layer_mrrs.append(float(m))
    best_layer     = int(np.argmax(layer_mrrs))
    best_layer_mrr = float(np.max(layer_mrrs))
    print(f'  Best layer: {best_layer} / {N_LAYERS}  (MRR={best_layer_mrr:.3f})')

    # ── Contrastive alignment ────────────────────────────────────────────────
    print('Contrastive alignment...')
    loader = DataLoader(PairDataset(train_ea, train_eb),
                        batch_size=64, shuffle=True)
    proj   = ProjectionHead(d_in=HIDDEN_SIZE).to(DEVICE)
    optim  = torch.optim.AdamW(proj.parameters(), lr=1e-4)
    proj.train()
    for epoch in range(10):
        total = 0.0
        for xa, xb in loader:
            xa, xb = xa.to(DEVICE), xb.to(DEVICE)
            loss   = nt_xent(proj(xa), proj(xb))
            optim.zero_grad(); loss.backward(); optim.step()
            total += loss.item()
        print(f'  epoch {epoch+1}/10  loss={total/len(loader):.4f}')
    proj.eval()
    with torch.no_grad():
        ea_c = proj(torch.tensor(ea, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        eb_c = proj(torch.tensor(eb, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    mrr_contr, _ = compute_mrr(ea_c, eb_c, correct_idx)
    print(f'  Contrastive MRR: {mrr_contr:.3f}')

    # ── Save ────────────────────────────────────────────────────────────────
    save(f'exp6/{lang}_layer_mrrs.json', layer_mrrs,
         description=f'Exp 6 layer-wise MRR — LaBSE {lang}')
    int_results[lang] = {
        'baseline_mrr'   : mrr_base,
        'normalised_mrr' : mrr_norm,
        'whitened_mrr'   : mrr_white,
        'csls_mrr'       : mrr_csls_val,
        'contrastive_mrr': mrr_contr,
        'best_layer'     : best_layer,
        'best_layer_mrr' : best_layer_mrr,
        'layer_mrrs'     : layer_mrrs,
    }
    save(f'exp6/{lang}_results.json', int_results[lang],
         description=f'Exp 6 full results — LaBSE {lang}')

    print(f'\n  Summary — {lang}')
    for k, v in [('Baseline',    mrr_base),    ('Normalised',  mrr_norm),
                 ('Whitened',    mrr_white),   ('CSLS',        mrr_csls_val),
                 ('Contrastive', mrr_contr),   ('Best layer',  best_layer_mrr)]:
        print(f'    {k:<14}: {v:.3f}')

save('exp6/labse_all_results.json', int_results,
     description='Exp 6 all intervention results — LaBSE')
print('\nExperiment 6 complete.')

## 13. Full results summary

Prints all tables in paper format for direct copy-paste.

In [ ]:
sep = '=' * 68
print(sep, 'LABSE RESULTS SUMMARY', sep, sep='\n')

print('\n--- Table 2 (LaBSE column): Layer-wise cosine similarity ---')
header = f'{"Language":<12}  ' + '  '.join(f'{int(f*100):3d}%' for f in LAYER_FRACS)
print(header)
for lang, res in layerwise.items():
    vals = '  '.join(f'{v["mean"]:.3f}' for v in res.values())
    print(f'{lang:<12}  {vals}')

print('\n--- Table 3 (LaBSE column): Script cluster silhouette ---')
for lang, res in cluster_res.items():
    print(f'  {lang:<12} {res["silhouette"]:.3f}')

print('\n--- Table 4 (LaBSE column): MLM script preference ---')
for lang, res in mlm_results.items():
    print(f'  {lang:<12} Script A pref = {res["prefer_a"]:.1%}  (n={res["n"]})')

print('\n--- Table 5 (LaBSE row): Cross-model comparison ---')
print(f'{"Language":<12}  {"FL cosim":>9}  {"MRR":>6}  {"Tok. overlap":>12}')
for lang in DATASETS:
    fl = layerwise[lang][max(layerwise[lang].keys())]['mean']
    print(f'{lang:<12}  {fl:>9.3f}  {retrieval_results[lang]:>6.3f}  {token_overlaps[lang]:>12.3f}')

print('\n--- Table 6 (LaBSE column): Retrieval MRR ---')
for lang, mrr in retrieval_results.items():
    print(f'  {lang:<12} {mrr:.3f}')
print(f'  {"Mean":<12} {np.mean(list(retrieval_results.values())):.3f}')

print('\n--- New Table 8: Intervention study (LaBSE) ---')
e6_rows = [
    {'Language'    : l,
     'Baseline'    : round(v['baseline_mrr'],    3),
     'Norm.'       : round(v['normalised_mrr'],  3),
     'Whitened'    : round(v['whitened_mrr'],    3),
     'CSLS'        : round(v['csls_mrr'],        3),
     'Contrastive' : round(v['contrastive_mrr'], 3),
     'Best layer'  : v['best_layer'],
     'Best MRR'    : round(v['best_layer_mrr'],  3)}
    for l, v in int_results.items()
]
e6_df = pd.DataFrame(e6_rows)
print(e6_df.to_string(index=False))

# Save full summary
full_summary = {
    'model'              : LABSE_MODEL,
    'exp1_layerwise'     : {l: {str(k): v for k,v in r.items()} for l,r in layerwise.items()},
    'exp2_silhouette'    : {l: r['silhouette'] for l,r in cluster_res.items()},
    'exp3_mlm'           : mlm_results,
    'exp4_mrr'           : retrieval_results,
    'exp4_token_overlap' : token_overlaps,
    'exp4_r_overlap_mrr' : r_mrr,
    'exp4_r_overlap_sim' : r_sim,
    'exp5_decomposition' : decomp_rows,
    'exp5_data_vol_sil'  : data_vol_sil,
    'exp5_data_vol_mrr'  : data_vol_mrr,
    'exp6_interventions' : e6_rows,
}
save('full_summary.json', full_summary,
     description='Complete LaBSE results summary — all experiments')

MANIFEST['completed'] = datetime.now(timezone.utc).isoformat()
_flush()

print(f'\n{sep}')
print(f'All outputs in : {RUN_DIR}')
print(f'Manifest       : {RUN_DIR}/manifest.json')
print(f'Files saved    : {len(MANIFEST["files"])}')
print(sep)

## 14. Layer-wise MRR plot (Experiment 6 supplement)

Shows which layer achieves peak retrieval MRR for each language — a useful
supplement to Table 8 that reveals whether LaBSE's final layer is optimal
(as expected from its retrieval-oriented fine-tuning) or whether mid-layer
representations are better, as observed for XLM-R.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for lang, res in int_results.items():
    lmrrs = res['layer_mrrs']
    ax.plot(range(len(lmrrs)), lmrrs,
            marker='o', markersize=4,
            label=lang_labels.get(lang, lang),
            color=lang_colors.get(lang, None))
ax.set_xlabel('Layer index (0 = embedding, 12 = final)')
ax.set_ylabel('Retrieval MRR')
ax.set_title('LaBSE: retrieval MRR by layer and language', fontweight='bold')
ax.legend(fontsize=8, ncol=2)
ax.set_xticks(range(N_LAYERS + 1))
plt.tight_layout()
savefig('exp6_layer_mrr_labse.pdf', fig,
        description='Exp 6 layer-wise MRR — LaBSE')
plt.show()

## Notes for paper integration

**Table 2:** Copy the LaBSE rows from Section 13 into the fifth model column.

**Table 3:** Copy the LaBSE silhouette column. Note that LaBSE has 768-dim
embeddings vs 1024-dim for the Large models — silhouette scores are not
directly comparable across dimensionalities but the pattern across languages is.

**Table 4:** Copy MLM preference. LaBSE covers more word pairs than XLM-R due
to its larger vocabulary; report the n values alongside preferences.

**Tables 5 & 6:** The LaBSE MRR row belongs in Table 6. For Table 5, note that
LaBSE's token overlap is computed with its own 501k-token tokenizer, which may
yield different overlap values than XLM-R's 250k tokenizer — report both or
note the tokenizer difference in the caption.

**New Table 8:** Present the LaBSE intervention results as a standalone table
parallel to Table 7. In Section 7, add a paragraph discussing whether the same
interventions are necessary or redundant given LaBSE's stronger baseline.

**Section 5.1:** Add LaBSE to the architecture discussion. Its translation-ranking
objective is qualitatively distinct from both masked-LM and span-corruption, so
it should be discussed in its own paragraph rather than grouped with the others.

**Section 5.3:** Update the correlation table with LaBSE's r values. If the
correlation holds (r > 0.7) it strengthens the model-agnostic diagnostic claim.

**Section 7.6 (recommendations):** If LaBSE outperforms mT5 on mean MRR, revise
the recommendation to: prefer LaBSE when cross-lingual retrieval is the target
task; prefer mT5 for other cross-script NLP tasks where no task-specific
fine-tuning data is available.